In [0]:
updates = [

(102,"Priya",90000),
(104,"Sneha",70000)

]

updates_df = spark.createDataFrame(
    updates,
    ["emp_id","name","salary"]
)

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(
    spark,
    "/tmp/employees_delta"
)

delta_table.alias("target") \
.merge(
    updates_df.alias("source"),
    "target.emp_id = source.emp_id"
) \
.whenMatchedUpdateAll() \
.whenNotMatchedInsertAll() \
.execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql DESCRIBE HISTORY employees

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-06-17T14:20:24.000Z,144170603529934,azuser7220_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(960343031370743),8128d6d3-b86d-44d7-8874-8737f4fbbef3,0617-141818-dgnkt220-v2n,null,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 3, numOutputBytes -> 1160)",null,Databricks-Runtime/18.2.x-photon-scala2.13


In [0]:
df = spark.read.format("delta") \
.option("versionAsOf",0) \
.load("/tmp/employees_delta")
 
display(df)
     

emp_id,name,salary
101,Rahul,75000
102,Priya,85000
103,Amit,65000


In [0]:
%sql OPTIMIZE employees
     

path,metrics
abfss://unity-catalog-storage@dbstorage7iipoczax2mti.dfs.core.windows.net/7405606344410216/__unitystorage/catalogs/57cabdc2-b77d-4164-aaff-2bc60c1111b4/tables/74244e5a-94cd-4832-8d40-72c3b988ef9a,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1781707041703, 1781707042338, 8, 0, null, List(0, 0), null, 3, 3, 0, 0, null, null)"


In [0]:
%sql OPTIMIZE employees
ZORDER BY(salary)
     

path,metrics
abfss://unity-catalog-storage@dbstorage7iipoczax2mti.dfs.core.windows.net/7405606344410216/__unitystorage/catalogs/57cabdc2-b77d-4164-aaff-2bc60c1111b4/tables/74244e5a-94cd-4832-8d40-72c3b988ef9a,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 1160), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1781707079041, 1781707079662, 8, 0, null, List(0, 0), null, 3, 3, 0, 0, null, null)"


In [0]:
%sql VACUUM employees

path
abfss://unity-catalog-storage@dbstorage7iipoczax2mti.dfs.core.windows.net/7405606344410216/__unitystorage/catalogs/57cabdc2-b77d-4164-aaff-2bc60c1111b4/tables/74244e5a-94cd-4832-8d40-72c3b988ef9a
